# FinGPT Crypto Drift Guard

**FinGPT is the decision-making agent.** It reads raw tweet text, infers sentiment, and decides BUY / SELL / HOLD. Full reasoning traces are stored at every step.

Three experiments measure how mutations change FinGPT's decisions:

| Experiment | What changes | Expected FinGPT effect |
|---|---|---|
| **Baseline** | — | Reference trajectory |
| **Intensity k=2** | Tweet feed is amplified by prepending emphasis | FinGPT may flip neutral → positive/negative |
| **Temporal Jitter n=3** | Tweets shifted 3 steps (stale news) | FinGPT reads wrong tweet → decision drift |

**Before running:** `Runtime → Change runtime type → T4 GPU`  
Upload `crypto_10k_tweets_(2021_2022Nov).csv` → `MyDrive/crypto-drift-guard/data/`

In [ ]:
# Cell 1 — Install dependencies
!pip install -q -U transformers peft bitsandbytes accelerate sentencepiece
print("Done")


In [ ]:
# Cell 2 — Verify GPU
import torch
assert torch.cuda.is_available(), "No GPU! Go to Runtime → Change runtime type → T4 GPU"
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Cell 3 — Mount Drive + configure paths
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_BASE  = '/content/drive/MyDrive/crypto-drift-guard'
TWEET_CSV   = f'{DRIVE_BASE}/data/crypto_10k_tweets_(2021_2022Nov).csv'
CACHE_PATH  = f'{DRIVE_BASE}/outputs/fingpt_sentiment_cache.json'
OUTPUTS_DIR = f'{DRIVE_BASE}/outputs'
os.makedirs(OUTPUTS_DIR, exist_ok=True)

assert os.path.exists(TWEET_CSV), (
    f"Dataset not found: {TWEET_CSV}\n"
    "Upload the CSV to MyDrive/crypto-drift-guard/data/"
)
print(f"Dataset : {TWEET_CSV}")
print(f"Outputs : {OUTPUTS_DIR}")

## Load FinGPT Model
Downloads `tiiuae/falcon-7b` (~14 GB) + `FinGPT/fingpt-mt_falcon-7b_lora`.  
~10 min on first run; model is cached for the Colab session.

In [ ]:
# Cell 4 — Load Flan-T5-Large
#
# Flan-T5-Large (770M params) is Google's instruction-tuned T5.
# No quantization needed — fits in float16 in ~1.5 GB VRAM.
# Handles the FinGPT BUY/HOLD/SELL decision prompt out of the box.

import os, gc
import torch
from transformers import T5ForConditionalGeneration, T5Tokenizer

MODEL_NAME = 'google/flan-t5-large'

# Clear any leftover allocations
torch.cuda.empty_cache()
gc.collect()

free_gb  = torch.cuda.mem_get_info()[0] / 1e9
total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'VRAM: {free_gb:.1f} GB free / {total_gb:.1f} GB total')

print('Loading Flan-T5-Large...')
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
model = T5ForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='auto',
)
model.eval()

used_gb = torch.cuda.memory_allocated() / 1e9
print(f'Model ready!  GPU in use: {used_gb:.1f} / {total_gb:.1f} GB')


In [ ]:
# Cell 5 — Two inference functions
#
# fingpt_infer  : tweet -> sentiment label   (pre-scoring, builds market features)
# fingpt_decide : tweet + RSI + volatility  -> BUY/HOLD/SELL  (FinGPT makes the decision)
#
# Note: Flan-T5 is encoder-decoder so we decode out[0] directly —
# no need to slice off the input length like decoder-only models.

def _run_model(prompt: str, max_new_tokens: int = 8) -> str:
    """Shared tokenise -> generate -> decode helper for T5."""
    inputs = tokenizer(
        prompt, return_tensors='pt', truncation=True, max_length=512
    ).to('cuda')
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens)
    return tokenizer.decode(out[0], skip_special_tokens=True).strip()

# ── Pre-scorer: tweet -> sentiment label (used to build RSI / vol features) ───
_SENTIMENT_PROMPT = (
    'Classify the sentiment of this cryptocurrency tweet. '
    'Answer with exactly one word: positive, neutral, or negative.\n'
    'Tweet: {text}'
)

def fingpt_infer(text: str) -> tuple[str, str]:
    """Tweet -> (sentiment_label, raw_output).  Used for pre-scoring only."""
    raw = _run_model(_SENTIMENT_PROMPT.format(text=text[:400]))
    low = raw.lower()
    if   'positive' in low: label = 'positive'
    elif 'negative' in low: label = 'negative'
    else:                   label = 'neutral'
    return label, raw

# ── Decision function: FinGPT receives tweet + market context, decides directly ─
# No hardcoded mapping — the model reasons over text + numbers together.
_DECISION_PROMPT = (
    'You are a cryptocurrency trading agent. '
    'Given the tweet and market conditions '
    '(RSI={rsi:.0f} where above 70 is overbought and below 30 is oversold, '
    'Volatility={vol:.0f} out of 100 where above 90 is extreme risk), '
    'decide the trading action. '
    'Answer with exactly one word: BUY, HOLD, or SELL.\n'
    'Tweet: {text}'
)

def fingpt_decide(text: str, rsi: float, vol: float) -> tuple[str, str]:
    """
    FinGPT directly decides the trading action from tweet + market context.
    Returns (action, raw_output) where action in {'BUY','HOLD','SELL'}.
    """
    prompt = _DECISION_PROMPT.format(text=text[:350], rsi=rsi, vol=vol)
    raw    = _run_model(prompt)
    low    = raw.lower()
    if   'buy'  in low: action = 'BUY'
    elif 'sell' in low: action = 'SELL'
    else:               action = 'HOLD'
    return action, raw

# ── Smoke tests ───────────────────────────────────────────────────────────────
tweet_bull = 'Bitcoin just hit a new ATH! This bull run is incredible! #BTC'
tweet_bear = 'Crypto market crashing. All my investments are in the red. #BTC'

print('=== fingpt_infer (sentiment pre-scorer) ===')
for tw in [tweet_bull, tweet_bear]:
    lbl, raw = fingpt_infer(tw)
    print(f'  [{lbl:>8}]  raw={repr(raw):<16}  tweet: {tw[:55]}')

print('\n=== fingpt_decide (model makes trading decision) ===')
tests = [
    (tweet_bull, 75.0, 25.0, 'RSI overbought -> may resist buying despite bullish tweet'),
    (tweet_bull, 45.0, 20.0, 'Normal RSI     -> likely BUY on bullish tweet'),
    (tweet_bear, 25.0, 30.0, 'RSI oversold   -> may BUY the dip despite bearish tweet'),
]
for tw, rsi, vol, note in tests:
    act, raw = fingpt_decide(tw, rsi, vol)
    print(f'  [{act:>4}]  RSI={rsi:.0f} Vol={vol:.0f}  {note}')
    print(f'          raw={repr(raw)}')


## Load Tweets & Pre-score with FinGPT
Scores are cached to Drive — re-runs are instant. Adjust `SAMPLE_SIZE` as needed.

In [ ]:
# Cell 6 — Load tweet dataset
import pandas as pd
import ast

SAMPLE_SIZE = 2000   # Increase to 10438 for full dataset (~3 h on T4)

df_raw = pd.read_csv(
    TWEET_CSV, on_bad_lines='skip', encoding='utf-8', engine='python'
)
df_raw['Date'] = pd.to_datetime(df_raw['Date'], utc=True, errors='coerce')
df_raw = (
    df_raw.dropna(subset=['Content'])
          .sort_values('Date')
          .reset_index(drop=True)
          .head(SAMPLE_SIZE)
)
print(f'Loaded {len(df_raw)} tweets')
print(df_raw[['Date', 'Content']].head(3).to_string())

In [ ]:
# Cell 7 — Pre-score all tweets with FinGPT (cached to Drive)
import json

# Load existing cache
cache: dict[str, str] = {}
if os.path.exists(CACHE_PATH):
    with open(CACHE_PATH) as f:
        cache = json.load(f)
    print(f'Cache loaded: {len(cache)} entries')

baseline_labels: list[str] = []   # label per row, aligned with df_raw
new_count = 0

for i, row in df_raw.iterrows():
    text = str(row['Content']).strip()[:400]
    if text in cache:
        baseline_labels.append(cache[text])
    else:
        lbl, _ = fingpt_infer(text)
        cache[text] = lbl
        baseline_labels.append(lbl)
        new_count += 1
        if new_count % 50 == 0:          # flush every 50 new scores
            with open(CACHE_PATH, 'w') as f:
                json.dump(cache, f)
    if (i + 1) % 200 == 0:
        print(f'  {i+1}/{len(df_raw)} scored  (new this run: {new_count})')

with open(CACHE_PATH, 'w') as f:         # final flush
    json.dump(cache, f)

dist = pd.Series(baseline_labels).value_counts().to_dict()
print(f'\nDone. New: {new_count}  Cached: {len(baseline_labels)-new_count}')
print(f'Label distribution: {dist}')

In [ ]:
# Cell 8 — Build market-compatible DataFrame from FinGPT scores

CRYPTO_MAP = {
    'bitcoin':'BTC', 'btc':'BTC', 'ethereum':'ETH', 'eth':'ETH',
    'bnb':'BNB',     'solana':'SOL', 'sol':'SOL',   'cardano':'ADA', 'ada':'ADA',
    'dogecoin':'DOGE','doge':'DOGE','xrp':'XRP',    'ripple':'XRP',
    'litecoin':'LTC','ltc':'LTC',   'polkadot':'DOT','dot':'DOT',
    'avalanche':'AVAX','avax':'AVAX',
}
SCORE_MAP = {'positive': 1.0, 'neutral': 0.0, 'negative': -1.0}

def extract_crypto(hashtags_str) -> str:
    if pd.isna(hashtags_str): return 'CRYPTO'
    try:
        for tag in ast.literal_eval(str(hashtags_str)):
            sym = CRYPTO_MAP.get(tag.lower().strip())
            if sym: return sym
    except Exception:
        pass
    return 'CRYPTO'

def build_market_df(tweet_df: pd.DataFrame, sentiment_labels: list) -> pd.DataFrame:
    """
    Build a market-compatible DataFrame from tweets + FinGPT labels.
    Derives RSI proxy, volatility proxy, and price signal from sentiment scores
    so the existing simulator and oracle can run unchanged.
    """
    df = tweet_df.copy().reset_index(drop=True)
    df['cryptocurrency']         = df['Hashtags'].apply(extract_crypto)
    df['tweet_text']             = df['Content'].str.strip()
    df['timestamp']              = df['Date']
    df['social_sentiment_score'] = [SCORE_MAP.get(l, 0.0) for l in sentiment_labels]
    # news_impact_score mirrors sentiment — used by temporal-jitter mutation
    df['news_impact_score']      = df['social_sentiment_score']
    s = df['social_sentiment_score']
    # RSI proxy: 14-step rolling mean of sentiment, scaled to [0, 100]
    df['rsi_technical_indicator']  = (50 + s.rolling(14, min_periods=1).mean() * 50).clip(0, 100)
    # Volatility proxy: 10-step rolling std of sentiment, scaled to [0, 100]
    df['volatility_index']         = (s.rolling(10, min_periods=1).std().fillna(0) * 100).clip(0, 100)
    # Price-change proxy: 1-step lagged sentiment × 3 %  (used in PnL simulation)
    df['price_change_24h_percent'] = s.shift(1).fillna(0) * 3.0
    df['current_price_usd']        = 100.0
    return df[[
        'timestamp', 'cryptocurrency', 'tweet_text',
        'social_sentiment_score', 'news_impact_score',
        'rsi_technical_indicator', 'volatility_index',
        'price_change_24h_percent', 'current_price_usd',
    ]]

baseline_df = build_market_df(df_raw, baseline_labels)
print(f'Baseline market DataFrame: {baseline_df.shape}')
print(baseline_df[['cryptocurrency','social_sentiment_score','rsi_technical_indicator','volatility_index']].describe().round(3))

## FinGPT Decision Agent

**Decision pipeline for each tweet:**
```
tweet_text
    │
    ▼
FinGPT inference  →  label (positive / neutral / negative)
    │
    ▼
Policy rule       →  policy_action (BUY / SELL / HOLD)
    │
    ▼
Risk Oracle       →  action_taken  (if volatility > 90 → force HOLD)
```

Every step is stored in a `FinGPTTrajectoryEntry` so you can trace exactly which tweet drove which decision and why.

In [ ]:
# Cell 9 — FinGPT Trajectory Entry
# Every field documents one step in the decision chain so you can trace exactly
# which tweet drove which action, what market context FinGPT saw, and why the
# oracle did or did not override.
from dataclasses import dataclass, field, asdict
from typing import List

@dataclass
class FinGPTTrajectoryEntry:
    step:               int     # row index in the experiment
    timestamp:          str
    cryptocurrency:     str
    # --- what FinGPT received ---
    tweet_text:         str     # exact tweet fed to the model
    context_rsi:        float   # RSI included in the decision prompt
    context_vol:        float   # volatility included in the decision prompt
    # --- what FinGPT decided ---
    fingpt_raw_output:  str     # verbatim model generation (untruncated)
    fingpt_action:      str     # BUY/HOLD/SELL parsed from raw output
    # --- risk oracle (hard override, independent of FinGPT) ---
    oracle_triggered:   bool    # True if oracle overrode FinGPT's decision
    action_taken:       str     # final action after oracle
    # --- simulation ---
    price_change_pct:   float   # used for PnL calculation
    # --- human-readable trace ---
    reasoning:          str     # full chain: context -> fingpt_action -> oracle -> action_taken

@dataclass
class FinGPTTrajectory:
    experiment: str
    entries: List[FinGPTTrajectoryEntry] = field(default_factory=list)

    def to_dataframe(self) -> pd.DataFrame:
        return pd.DataFrame([asdict(e) for e in self.entries])

    def save(self, path: str):
        self.to_dataframe().to_json(path, orient='records', indent=2)
        print(f'  Saved: {path}')

print('FinGPTTrajectoryEntry defined')


In [ ]:
# Cell 10 — FinGPT Decision Agent
#
# FinGPT is the decision maker. For each row it:
#   1. Reads tweet_text from the (possibly mutated) DataFrame
#   2. Reads RSI + volatility from the same row (market context for the prompt)
#   3. Calls fingpt_decide(tweet, rsi, vol) -> FinGPT outputs BUY/HOLD/SELL
#   4. Applies the Risk Oracle as a hard safety cap  (vol > 90 -> force HOLD)
#   5. Stores every detail in a FinGPTTrajectoryEntry
#
# There is NO sentiment->action mapping anywhere in this class.

VOLATILITY_ORACLE_LIMIT = 90.0

class FinGPTDecisionAgent:
    def __init__(self):
        # Decision cache keyed by (tweet, rsi_bucket, vol_bucket).
        # Bucketing avoids redundant API calls for near-identical contexts.
        self._dcache: dict[str, str] = {}

    def _decide(self, tweet: str, rsi: float, vol: float) -> tuple[str, str]:
        """Ask FinGPT: given this tweet + market context, BUY/HOLD/SELL?"""
        rsi_b = round(rsi / 5) * 5    # bucket to nearest 5
        vol_b = round(vol / 5) * 5
        key   = f'{tweet[:400]}|rsi={rsi_b}|vol={vol_b}'
        if key in self._dcache:
            return self._dcache[key], f'[cached] {self._dcache[key]}'
        action, raw = fingpt_decide(tweet, rsi, vol)
        self._dcache[key] = action
        return action, raw

    def _oracle(self, vol: float, action: str) -> tuple[str, bool]:
        """Hard risk override: extreme volatility forces HOLD regardless."""
        if vol > VOLATILITY_ORACLE_LIMIT:
            return 'HOLD', True
        return action, False

    def run(self, df: pd.DataFrame, experiment_name: str) -> FinGPTTrajectory:
        traj = FinGPTTrajectory(experiment=experiment_name)
        n    = len(df)
        live = 0

        for i, (_, row) in enumerate(df.iterrows()):
            tweet = str(row.get('tweet_text', '')).strip()[:400]
            rsi   = float(row['rsi_technical_indicator'])
            vol   = float(row['volatility_index'])
            pct   = float(row['price_change_24h_percent'])

            # FinGPT decides (from tweet text + market context in prompt)
            fg_action, raw_out = self._decide(tweet, rsi, vol)
            if '[cached]' not in raw_out:
                live += 1

            # Oracle hard-override (safety layer, not a soft policy)
            final_action, oracle_hit = self._oracle(vol, fg_action)

            reason = (
                f'FinGPT saw RSI={rsi:.0f} Vol={vol:.0f} '
                f'-> decided={fg_action} (raw: {repr(raw_out[:40])})'
            )
            if oracle_hit:
                reason += f' -> ORACLE overrode (vol={vol:.1f}>90) -> HOLD'
            reason += f' | tweet: "{tweet[:70]}"'

            traj.entries.append(FinGPTTrajectoryEntry(
                step              = i,
                timestamp         = str(row['timestamp']),
                cryptocurrency    = str(row['cryptocurrency']),
                tweet_text        = tweet,
                context_rsi       = rsi,
                context_vol       = vol,
                fingpt_raw_output = raw_out,
                fingpt_action     = fg_action,
                oracle_triggered  = oracle_hit,
                action_taken      = final_action,
                price_change_pct  = pct,
                reasoning         = reason,
            ))

            if (i + 1) % 200 == 0 or (i + 1) == n:
                print(f'  [{experiment_name}] {i+1}/{n}  live FinGPT calls={live}')

        return traj

print('FinGPTDecisionAgent defined')


In [ ]:
# Cell 11 — Simulator + drift analysis helpers

POSITION_SIZE = 1000.0
DIRECTION     = {'BUY': 1, 'SELL': -1, 'HOLD': 0}

@dataclass
class PnLResult:
    total_pnl:   float
    trade_count: int
    win_count:   int
    loss_count:  int
    win_rate:    float
    detail:      pd.DataFrame

def simulate(traj: 'FinGPTTrajectory') -> PnLResult:
    df = traj.to_dataframe().copy()
    direction    = df['action_taken'].map(DIRECTION)
    df['pnl']    = direction * (df['price_change_pct'] / 100.0) * POSITION_SIZE
    df['cum_pnl']= df['pnl'].cumsum()
    trades  = df[df['action_taken'] != 'HOLD']
    wins    = (trades['pnl'] > 0).sum()
    losses  = (trades['pnl'] < 0).sum()
    n       = len(trades)
    return PnLResult(
        total_pnl=float(df['pnl'].sum()), trade_count=n,
        win_count=int(wins), loss_count=int(losses),
        win_rate=wins/n if n else 0.0, detail=df,
    )

def pnl_line(label: str, r: PnLResult) -> str:
    return (f'  {label:<24} | P&L: {r.total_pnl:>+9.2f} USD'
            f' | Trades: {r.trade_count:>4} | Win rate: {r.win_rate*100:>5.1f}%')

def drift_report(base_traj: FinGPTTrajectory, mut_traj: FinGPTTrajectory) -> dict:
    """Compare FinGPT decisions between baseline and a mutated trajectory."""
    b = base_traj.to_dataframe()
    m = mut_traj.to_dataframe()
    total     = len(b)
    mismatch  = (b['action_taken'] != m['action_taken']).sum()
    oracle_b  = (b['oracle_triggered'] == True).sum()
    oracle_m  = (m['oracle_triggered'] == True).sum()
    # Flip table: which (baseline_action → mutant_action) transitions happened?
    gap_mask  = b['action_taken'] != m['action_taken']
    flips     = (
        b['action_taken'][gap_mask] + ' → ' + m['action_taken'][gap_mask]
    ).value_counts().to_dict()
    return dict(
        total=total, mismatch=int(mismatch),
        adr=mismatch/total if total else 0.0,
        oracle_baseline=int(oracle_b), oracle_mutant=int(oracle_m),
        action_flips=flips,
    )

def print_drift(report: dict, label: str):
    print(f'  Experiment        : {label}')
    print(f'  Total steps       : {report["total"]}')
    print(f'  Decision changes  : {report["mismatch"]}  (ADR = {report["adr"]*100:.2f}%)')
    print(f'  Oracle (baseline) : {report["oracle_baseline"]}')
    print(f'  Oracle (mutant)   : {report["oracle_mutant"]}')
    print(f'  Action flips      : {report["action_flips"]}')

print('Simulator + drift helpers defined')

In [ ]:
# Cell 12 — Mutation operators

def mutate_intensity(df: pd.DataFrame, factor: float = 2.0) -> pd.DataFrame:
    """
    Intensity Mutation (k=2): prepends emphasis marker to each tweet.

    Why text-based, not numeric?
    FinGPT reads the raw tweet, so amplifying only a numeric score would leave
    the model's input unchanged. Instead, we prepend '[BREAKING]' to every tweet,
    simulating an amplified, high-urgency feed. This can push neutral tweets
    toward positive or negative depending on context.

    The derived market features (RSI, volatility) are also recalculated from the
    numeric score (score × factor, clipped) so the oracle sees an amplified signal.
    """
    m = df.copy()
    # Amplify numeric score for oracle-derived features
    m['social_sentiment_score'] = (m['social_sentiment_score'] * factor).clip(-1.0, 1.0)
    s = m['social_sentiment_score']
    m['rsi_technical_indicator']  = (50 + s.rolling(14, min_periods=1).mean() * 50).clip(0, 100)
    m['volatility_index']         = (s.rolling(10, min_periods=1).std().fillna(0) * 100).clip(0, 100)
    m['price_change_24h_percent'] = s.shift(1).fillna(0) * 3.0
    # Amplify tweet text: prepend emphasis marker so FinGPT sees a louder signal
    m['tweet_text'] = '[BREAKING] ' + m['tweet_text'].astype(str)
    return m

def mutate_temporal_jitter(df: pd.DataFrame, shift: int = 3) -> pd.DataFrame:
    """
    Temporal Jitter (n=3): shift the tweet feed by `shift` rows.

    At step T the agent now reads the tweet that was originally at step T-shift.
    This simulates news delivery lag — the agent acts on stale information.
    The first `shift` rows receive a neutral placeholder tweet.
    Market features are also shifted to maintain consistency.
    """
    m = df.copy()
    # Shift tweets (agent reads older tweets)
    shifted_tweets = m['tweet_text'].shift(shift)
    m['tweet_text'] = shifted_tweets.fillna('[NO DATA — tweet feed not yet started]')
    # Shift numeric signals too (consistent with news lag)
    m['news_impact_score'] = m['news_impact_score'].shift(shift).fillna(
        df['news_impact_score'].mean()
    )
    return m

print('Mutation operators defined')

## Run All Three FinGPT Experiments

In [ ]:
# Cell 13 — Run baseline + both mutations
SEP = '=' * 65

agent = FinGPTDecisionAgent()

# ── Baseline ──────────────────────────────────────────────────────────────────
print(f'{SEP}\n  EXPERIMENT 1: Baseline\n{SEP}')
traj_baseline = agent.run(baseline_df, 'baseline')
sim_baseline  = simulate(traj_baseline)
print(pnl_line('FinGPT Baseline', sim_baseline))
traj_baseline.save(f'{OUTPUTS_DIR}/traj_fingpt_baseline.json')

# ── Intensity k=2 ─────────────────────────────────────────────────────────────
print(f'\n{SEP}\n  EXPERIMENT 2: Intensity Mutation (k=2)\n{SEP}')
df_intensity  = mutate_intensity(baseline_df, factor=2.0)
traj_intensity= agent.run(df_intensity, 'intensity_k2')
sim_intensity = simulate(traj_intensity)
print(pnl_line('FinGPT Intensity k=2', sim_intensity))
traj_intensity.save(f'{OUTPUTS_DIR}/traj_fingpt_intensity_k2.json')

# ── Temporal Jitter n=3 ───────────────────────────────────────────────────────
print(f'\n{SEP}\n  EXPERIMENT 3: Temporal Jitter (n=3)\n{SEP}')
df_jitter     = mutate_temporal_jitter(baseline_df, shift=3)
traj_jitter   = agent.run(df_jitter, 'temporal_jitter_n3')
sim_jitter    = simulate(traj_jitter)
print(pnl_line('FinGPT Jitter n=3', sim_jitter))
traj_jitter.save(f'{OUTPUTS_DIR}/traj_fingpt_temporal_jitter_n3.json')

print(f'\n{SEP}\n  ALL EXPERIMENTS COMPLETE\n{SEP}')

## Drift Report — How Much Did Mutations Change FinGPT's Decisions?

In [ ]:
# Cell 14 — Drift analysis: Baseline vs Mutations  (output saved to Drive)
import io, contextlib

def capture(fn):
    """Run fn(), return everything it printed as a string."""
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        fn()
    return buf.getvalue()

def run_drift_report():
    print(f'{SEP}')
    print(f'  DRIFT REPORT: Baseline vs Intensity k=2')
    print(f'{SEP}')
    print_drift(dr_intensity, 'intensity_k2')

    print(f'\n{SEP}')
    print(f'  DRIFT REPORT: Baseline vs Temporal Jitter n=3')
    print(f'{SEP}')
    print_drift(dr_jitter, 'temporal_jitter_n3')

    print(f'\n{SEP}')
    print(f'  P&L COMPARISON')
    print(f'{SEP}')
    print(pnl_line('Baseline',      sim_baseline))
    print(pnl_line('Intensity k=2', sim_intensity))
    print(pnl_line('Jitter n=3',    sim_jitter))
    print(f'  Intensity delta : {sim_intensity.total_pnl - sim_baseline.total_pnl:>+.2f} USD')
    print(f'  Jitter delta    : {sim_jitter.total_pnl    - sim_baseline.total_pnl:>+.2f} USD')

dr_intensity = drift_report(traj_baseline, traj_intensity)
dr_jitter    = drift_report(traj_baseline, traj_jitter)

report_text = capture(run_drift_report)
print(report_text)

# Save to Drive
save_path = f'{OUTPUTS_DIR}/drift_report.txt'
with open(save_path, 'w') as f:
    f.write(report_text)
print(f'Saved -> {save_path}')


## Trajectory Inspector
Browse the stored trace to understand **why** FinGPT made each decision.

In [ ]:
# Cell 15 — Inspect FinGPT decision trajectories

def show_decisions(traj: FinGPTTrajectory, n: int = 10, action_filter: str = None):
    """
    Print trajectory entries.
    action_filter: 'BUY' / 'SELL' / 'HOLD'  to see only those steps.
    """
    print(f'\n=== Trajectory: {traj.experiment} ===')
    shown = 0
    for e in traj.entries:
        if action_filter and e.action_taken != action_filter:
            continue
        print(f'  Step {e.step:>4} | {e.cryptocurrency:<6} '
              f'| fingpt_decided={e.fingpt_action:<4} '
              f'| oracle={e.oracle_triggered} '
              f'| final={e.action_taken}')
        print(f'           Tweet    : {e.tweet_text[:90]}')
        print(f'           Context  : RSI={e.context_rsi:.0f}  Vol={e.context_vol:.0f}')
        print(f'           Raw out  : {repr(e.fingpt_raw_output)}')
        print(f'           Reasoning: {e.reasoning}')
        print()
        shown += 1
        if shown >= n:
            break
    if shown == 0:
        print(f'  (no entries matching filter={action_filter})')

# Show first 5 BUY decisions from baseline
show_decisions(traj_baseline, n=5, action_filter='BUY')

# Show first 5 SELL decisions from baseline
show_decisions(traj_baseline, n=5, action_filter='SELL')


In [ ]:
# Cell 16 — Compare specific steps across experiments (spot the drift)

def compare_step(step: int):
    """Show the same step across all three experiments side-by-side."""
    trajs = [
        ('baseline',      traj_baseline),
        ('intensity_k2',  traj_intensity),
        ('jitter_n3',     traj_jitter),
    ]
    print(f'\n=== Step {step} across experiments ===')
    for name, traj in trajs:
        if step >= len(traj.entries):
            print(f'  {name}: step out of range')
            continue
        e = traj.entries[step]
        print(f'  [{name:<18}]  action={e.action_taken:<4}  label={e.fingpt_label}')
        print(f'    tweet: {e.tweet_text[:90]}')
        print(f'    why  : {e.reasoning}')
        print()

# Steps where baseline and jitter might disagree — pick any step
for step in [3, 10, 50]:
    compare_step(step)

## Plots

In [ ]:
# Cell 17 — Plots
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# ── Plot 1: Cumulative P&L across all three experiments ───────────────────────
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(sim_baseline.detail['cum_pnl'].values,  label='Baseline',        lw=2.5, color='steelblue')
ax.plot(sim_intensity.detail['cum_pnl'].values, label='Intensity k=2',   lw=2,   color='darkorange',  ls='--')
ax.plot(sim_jitter.detail['cum_pnl'].values,    label='Temporal Jitter n=3', lw=2, color='crimson', ls=':')
ax.axhline(0, color='black', lw=0.5, ls='-')
ax.set_title('FinGPT Agent — Cumulative P&L: Baseline vs Mutations', fontsize=13, fontweight='bold')
ax.set_xlabel('Step'); ax.set_ylabel('Cumulative P&L (USD)')
ax.legend(fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:+.0f}'))
plt.tight_layout()
plt.savefig(f'{OUTPUTS_DIR}/plot_fingpt_pnl_drift.png', dpi=150)
plt.show()

# ── Plot 2: Action distribution per experiment ────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
colors = {'BUY': '#2ecc71', 'SELL': '#e74c3c', 'HOLD': '#95a5a6'}
for ax, (traj, sim, title) in zip(axes, [
    (traj_baseline,  sim_baseline,  'Baseline'),
    (traj_intensity, sim_intensity, 'Intensity k=2'),
    (traj_jitter,    sim_jitter,    'Temporal Jitter n=3'),
]):
    counts = pd.Series([e.action_taken for e in traj.entries]).value_counts()
    bar_colors = [colors.get(a, 'grey') for a in counts.index]
    counts.plot.bar(ax=ax, color=bar_colors, edgecolor='white', width=0.6)
    ax.set_title(f'{title}\nP&L: ${sim.total_pnl:+.2f}', fontsize=10, fontweight='bold')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=0)
axes[0].set_ylabel('Count')
fig.suptitle('FinGPT Action Distribution per Experiment', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUTS_DIR}/plot_fingpt_action_dist.png', dpi=150)
plt.show()

# ── Plot 3: ADR bar (how much each mutation drifted FinGPT's decisions) ───────
fig, ax = plt.subplots(figsize=(6, 4))
labels_bar = ['Intensity k=2', 'Temporal Jitter n=3']
adrs       = [dr_intensity['adr'] * 100, dr_jitter['adr'] * 100]
bar_c      = ['#e67e22', '#c0392b']
bars = ax.bar(labels_bar, adrs, color=bar_c, edgecolor='white', width=0.4)
ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=12)
ax.set_ylabel('Action Difference Ratio (%)')
ax.set_title('Logic Drift: how much mutations changed\nFinGPT\'s decisions vs baseline', fontweight='bold')
ax.set_ylim(0, max(adrs) * 1.3 + 5)
plt.tight_layout()
plt.savefig(f'{OUTPUTS_DIR}/plot_fingpt_adr.png', dpi=150)
plt.show()

print(f'Plots saved to {OUTPUTS_DIR}/')

In [ ]:
# Cell 18 — Final summary
print('=' * 65)
print('  FINAL SUMMARY — FinGPT Decision Agent')
print('=' * 65)
print(f'  Model   : {BASE_MODEL} + {LORA_ADAPTER}')
print(f'  Tweets  : {len(df_raw)}')
print()
print('  DRIFT (Action Difference Ratio)')
print(f'    Intensity k=2         : {dr_intensity["adr"]*100:.2f}%  ({dr_intensity["mismatch"]} steps changed)')
print(f'    Temporal Jitter n=3   : {dr_jitter["adr"]*100:.2f}%  ({dr_jitter["mismatch"]} steps changed)')
print()
print('  P&L')
print(pnl_line('Baseline',      sim_baseline))
print(pnl_line('Intensity k=2', sim_intensity))
print(pnl_line('Jitter n=3',    sim_jitter))
print()
print(f'  All outputs in: {OUTPUTS_DIR}')
print('  Trajectory JSONs: traj_fingpt_baseline.json')
print('                    traj_fingpt_intensity_k2.json')
print('                    traj_fingpt_temporal_jitter_n3.json')
print('=' * 65)